In this notebook we parse texts on "Government Policies and Programs" extracted from USGS PDF Reports to construct a structured panel data capturing temporal and spatial policy variations. The unit of analysis in the output panel is `country-year(date)-policy`, with properties including, *inter alia*:

0. policy title with reference number (ref. ID) ([NAME], *str*);
1. administrative level (state/local, *categorical*);
2. issuing authority ([NAME], *str*); 
3. legal hierarchy ([TYPE_OF_LEGAL_DOCUMENT], *str*); 
4. policy domain (export_import_restriction, foreign_investment_access, equity_structure_state_ownership, technological_transfer, domestic_content_requirement, mining_concession, land_concession, tax_royalty, subsidy, price_control, labor_standard, environmental_standard, community_interest, foreign_exchange, geodata_secrecy, comprehensive_framework), *mulitiple choices concatenated to a list*);
5. policy content (*str*)
5. targeted minerals ([NAME], *list*); 
6. contents of the policy ([REGULATORY_TARGET_WIZ_DUTIES], *str*)

Given the lack of training data, there are two *prema facie* technical pathways at our disposal: (1) Purely based on state-of-the-art (SOTA) LLM with few-shot or zero-shot prompt engineering. (2) Hybrid of SOTA-LLM and BERT, where SOTA-LLM is appealed to for generating a training set, before BERT is invoked to accelerate coding process after pre-training.

In [1]:
import os
import sys
sys.path.append("../../modules/text_parsing/")

In [4]:
continent = 'asia'
country_year_policy = 'Philippines_2022_policy'
txt_path = "../../data/policy_section_txt/usgs_" + continent + '/' + country_year_policy + '.txt'
with open(txt_path, "r", encoding='utf-8') as file:
    input_text = file.read()
print(input_text)

The principal laws and regulations that regulate the mineral industry in the Philippines are the Philippine Mining Act of 1995 (Mining Act), the Clean Air Act of 1999, the Clean Water Act of 2004, and the Department of Environment and Natural Resources (DENR) Administrative Order No. 2010–21.  The Oil Exploration and Development Act (Presidential Decree No. 87) of 1972 mandates that the Government undertake petroleum exploration and production. The Coal Development Act (Presidential Decree No. 972) of 1976 mandates that the Government undertake active exploration, development, and production of coal resources or execute coal operating contracts. The DENR canceled 73 exploration-stage projects in February 2017, and in April 2017, announced (by DAO No. 2017–10) a ban on new open pit mining after issuing a permanent closure order to 22 of 41 operating open pit mines that had failed to pass audits for environmental violations (Department of Energy, 1972; Deutsche Welle, 2017; Philippine Ex

In [10]:
# Concatenate policy sections from independent reports concerning different years of the same country into a single string, before initiating a one-time impartment to the API
continent = 'asia'
country = 'Indonesia'
dir_path = "../../data/policy_section_txt/usgs_" + continent + '/'
files_list = [filename for filename in os.listdir(dir_path) if country in filename]
input_text = ''
for filename in files_list:
    file_path = "../../data/policy_section_txt/usgs_" + continent + '/' + filename
    with open(file_path, 'r', encoding='utf-8') as file:
        current_text = file.read()
        input_text = input_text + f'[THE SEGMENT BELOW IS FROM LOCAL FILE: {filename}]' + '\n'
        input_text = input_text + current_text
        input_text = input_text + '\n\n'
print(input_text)

[THE SEGMENT BELOW IS FROM LOCAL FILE: Indonesia_2005_policy.txt]
Indonesia’s principal mineral resources are coal, copper, gold, nickel, oil, and tin. Mining provides significant local employment directly at the mine site and indirectly through the supply of goods and services from Indonesian sources. A  significant part of Indonesia remains unexplored; some of these land areas are the most prospective for mineral development in the Asia and the Pacific region. The mining sector could have the potential to become a much larger contributor to the country’s economy and to regional development. The Government announced an “Investment Climate Improvement Package” containing 85 regulatory and institutional reforms to improve the investment climate; it was expected to take effect in 2006. The package included improving investment licensing and permitting procedures and synchronizing regulations and procedures between the Central Government and local administration. The Government planned va

In [5]:
import LLMStructuredOutput as lso
import os
if len(input_text.strip()) > 0:
    response = lso.qwen_structured(input_text = input_text, # Input text
                                   base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1",
                                   api_key = os.getenv("DASHSCOPE_API_KEY"), # API-KEY required, drawn from local environment variable
                                   model = "qwen3.6-flash")
else: 
    print("Current input is an empty string!")

In [6]:
from pydantic import BaseModel
response_dict = response.model_dump()
response_dict
## DEBUG: Script below obsolete in August 2026 due to major update of Qwen-API.
# import json; response_json = json.loads(response) # Turn the returned string into JSON-schema dictionary 

{'policies': [{'document_name': 'Oil Exploration and Development Act (Presidential Decree No. 87)',
   'reference_id': 'Presidential Decree No. 87',
   'issuing_year': 1972,
   'issuing_month': None,
   'issuing_day': None,
   'targeted_mineral': ['petroleum'],
   'policy_domain': ['mining_concession'],
   'policy_content': 'mandates that the Government undertake petroleum exploration and production.',
   'legal_hierarchy': 'decree',
   'issuing_authority': None,
   'is_local': False,
   'is_amendment': False,
   'amendment_target': None,
   'is_implement': False,
   'implemment_target': None},
  {'document_name': 'Coal Development Act (Presidential Decree No. 972)',
   'reference_id': 'Presidential Decree No. 972',
   'issuing_year': 1976,
   'issuing_month': None,
   'issuing_day': None,
   'targeted_mineral': ['coal'],
   'policy_domain': ['mining_concession'],
   'policy_content': 'mandates that the Government undertake active exploration, development, and production of coal resour

In [ ]:
import json
json_path = '../../data/policy_parsing_json/usgs_' + continent + '/' + country_year_policy + '.json' # Output path (for independent text)
# json_path = '../../data/policy_parsing_json/usgs_' + continent + '/' + country + '.json' # Output path (for concatenated text)
with open(json_path, 'w', encoding='utf-8') as file:
    json.dump(response_dict, file, indent=4, ensure_ascii=False)